In [5]:
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import astropy.units as u
from astropy.coordinates import GeocentricMeanEcliptic, get_sun
from astropy.time import Time

In [6]:
# --- Parameters (edit and re-run all) ---
R_EARTH_KM = 6378.137
# ALT_KM = 600.0

DEFAULT_ORBIT_ELEMENTS = {
    "inclination_deg": 97.8038,
    "raan_deg": 62.0432,
    "eccentricity": 0.0006523,
    "arg_perigee_deg": 113.5826,
    "mean_anomaly_deg": 187.8102,
}

In [7]:
def _unit(v: np.ndarray) -> np.ndarray:
    n = float(np.linalg.norm(v))
    if n <= 0.0:
        raise ValueError('zero-length vector')
    return v / n

EARTH_OBLIQUITY_DEG = 23.439281
# Plotting-frame convention used here:
# - +Z is ecliptic north
# - +X/+Y span the ecliptic plane
# - the Sun direction is computed from the actual geocentric ecliptic position
#   at the current UTC time, rather than being fixed at +X
# - Earth spin axis is tilted from ecliptic north by the obliquity
ECLIPTIC_NORTH = _unit(np.array([0.0, 0.0, 1.0], dtype=float))
EARTH_SPIN_AXIS = _unit(
    np.array([0.0, -np.sin(np.deg2rad(EARTH_OBLIQUITY_DEG)), np.cos(np.deg2rad(EARTH_OBLIQUITY_DEG))], dtype=float)
)


def _sun_hat_now_ecliptic(time_utc: str | Time | None = None) -> tuple[np.ndarray, Time, float, float]:
    """Return geocentric Sun direction in the notebook frame.

    When ``time_utc`` is ``None`` the current UTC time is used. String inputs
    are parsed by ``astropy.time.Time``.
    """
    if time_utc is None:
        t = Time.now()
    elif isinstance(time_utc, Time):
        t = time_utc
    else:
        t = Time(str(time_utc), scale='utc')
    sun_ecl = get_sun(t).transform_to(GeocentricMeanEcliptic(equinox=t))
    lon = float(sun_ecl.lon.to_value(u.rad))
    lat = float(sun_ecl.lat.to_value(u.rad))
    sun_hat = _unit(
        np.array(
            [
                np.cos(lat) * np.cos(lon),
                np.cos(lat) * np.sin(lon),
                np.sin(lat),
            ],
            dtype=float,
        )
    )
    return sun_hat, t, float(np.rad2deg(lon)) % 360.0, float(np.rad2deg(lat))


def _great_circle_points_from_normal(normal_hat: np.ndarray, npts: int = 721) -> np.ndarray:
    """Return unit-sphere points for the great circle with plane normal normal_hat."""
    normal_hat = _unit(np.asarray(normal_hat, dtype=float))
    ref = np.array([1.0, 0.0, 0.0], dtype=float)
    if abs(float(np.dot(ref, normal_hat))) > 0.9:
        ref = np.array([0.0, 1.0, 0.0], dtype=float)
    e1 = _unit(ref - normal_hat * float(np.dot(ref, normal_hat)))
    e2 = _unit(np.cross(normal_hat, e1))
    tt = np.linspace(0.0, 2.0 * np.pi, npts)
    return np.cos(tt)[:, None] * e1[None, :] + np.sin(tt)[:, None] * e2[None, :]

def _target_from_radec(ra_deg: float, dec_deg: float) -> np.ndarray:
    """Target unit vector from ICRS-like RA/Dec in the plotting frame.

    RA is measured in the XY plane from +X toward +Y.
    Dec is measured from the XY plane toward +Z.
    """
    ra = np.deg2rad(float(ra_deg))
    dec = np.deg2rad(float(dec_deg))
    x = np.cos(dec) * np.cos(ra)
    y = np.cos(dec) * np.sin(ra)
    z = np.sin(dec)
    return _unit(np.array([x, y, z], dtype=float))


def _solve_kepler_eccentric_anomaly(M_rad: float, ecc: float, max_iter: int = 30) -> float:
    """Solve Kepler's equation M = E - e sin E via Newton iterations."""
    E = M_rad if ecc < 0.8 else np.pi
    for _ in range(max_iter):
        f = E - ecc * np.sin(E) - M_rad
        fp = 1.0 - ecc * np.cos(E)
        dE = f / fp
        E -= dE
        if abs(float(dE)) < 1e-12:
            break
    return float(E)


def _sc_state_from_elements(alt_km: float, mean_anomaly_offset_deg: float = 0.0, elements: dict | None = None) -> np.ndarray:
    """Return spacecraft ECI position vector [km] from Keplerian elements.

    Uses semi-major axis a = R_EARTH_KM + alt_km as a practical approximation.
    """
    if elements is None:
        elements = DEFAULT_ORBIT_ELEMENTS

    inc = np.deg2rad(float(elements['inclination_deg']))
    raan = np.deg2rad(float(elements['raan_deg']))
    ecc = float(elements['eccentricity'])
    argp = np.deg2rad(float(elements['arg_perigee_deg']))
    M0 = float(elements['mean_anomaly_deg'])
    M = np.deg2rad(M0 + float(mean_anomaly_offset_deg))

    # Wrap mean anomaly to [-pi, pi] for stable Newton solve.
    M = float((M + np.pi) % (2.0 * np.pi) - np.pi)

    a = float(R_EARTH_KM + alt_km)
    E = _solve_kepler_eccentric_anomaly(M, ecc)

    cosE = np.cos(E)
    sinE = np.sin(E)
    r = a * (1.0 - ecc * cosE)

    # True anomaly from eccentric anomaly.
    nu = 2.0 * np.arctan2(np.sqrt(1.0 + ecc) * np.sin(E / 2.0), np.sqrt(1.0 - ecc) * np.cos(E / 2.0))

    # Perifocal coordinates.
    x_pf = r * np.cos(nu)
    y_pf = r * np.sin(nu)
    z_pf = 0.0
    r_pf = np.array([x_pf, y_pf, z_pf], dtype=float)

    # Rotation Q_pX = R3(raan) * R1(inc) * R3(argp)
    cO, sO = np.cos(raan), np.sin(raan)
    ci, si = np.cos(inc), np.sin(inc)
    cw, sw = np.cos(argp), np.sin(argp)

    Q = np.array([
        [cO * cw - sO * sw * ci, -cO * sw - sO * cw * ci, sO * si],
        [sO * cw + cO * sw * ci, -sO * sw + cO * cw * ci, -cO * si],
        [sw * si, cw * si, ci],
    ], dtype=float)

    return Q @ r_pf


def _compute_pr7_n_hat(sc_vec_km: np.ndarray, target_hat: np.ndarray) -> np.ndarray:
    """PR#7 nearest-limb surface normal for this SC+target geometry.

    Use the instantaneous spacecraft radius |r_sc|, not a nominal altitude,
    so the limb angle remains correct for the slightly eccentric orbit.
    """
    zenith = _unit(sc_vec_km)
    proj = target_hat - zenith * float(np.dot(target_hat, zenith))
    if float(np.linalg.norm(proj)) < 1e-12:
        ref = np.array([1.0, 0.0, 0.0], dtype=float)
        if abs(float(np.dot(ref, zenith))) > 0.9:
            ref = np.array([0.0, 1.0, 0.0], dtype=float)
        proj = ref - zenith * float(np.dot(ref, zenith))
    limb_dir = _unit(proj)

    r_sc = float(np.linalg.norm(sc_vec_km))
    limb_angle = float(np.arccos(np.clip(R_EARTH_KM / r_sc, -1.0, 1.0)))
    return _unit(np.cos(limb_angle) * zenith + np.sin(limb_angle) * limb_dir)


def _fractions(sc_hat: np.ndarray, sun_hat: np.ndarray, target_hat: np.ndarray, alt_km: float, n_samples: int = 30000):
    """Area fractions on unit sphere for visibility/daylight/target-direction gates."""
    r_sc = float(np.linalg.norm(_sc_state_from_elements(alt_km, 0.0)))
    thr = float(R_EARTH_KM / r_sc)
    sc_u = _sc_state_from_elements(alt_km, 0.0) / R_EARTH_KM

    rng = np.random.default_rng(0)
    n = rng.normal(size=(n_samples, 3))
    n /= np.linalg.norm(n, axis=1, keepdims=True)

    visible = (n @ sc_hat) >= thr
    dayside = (n @ sun_hat) >= 0.0

    d = n - sc_u[None, :]
    d /= np.linalg.norm(d, axis=1, keepdims=True)
    toward_target = (d @ target_hat) >= 0.0

    return {
        'visible': float(visible.mean()),
        'dayside_visible': float((visible & dayside).mean()),
        'dayside_visible_toward_target': float((visible & dayside & toward_target).mean()),
    }


def _panel_mask(i: int, j: int, sc_hat: np.ndarray, sc_u: np.ndarray, sun_hat: np.ndarray, target_hat: np.ndarray, thr: float, ngrid: int = 260):
    """Projected mask for dayside∩SC-visible∩toward-target."""
    k = 3 - i - j
    u = np.linspace(-1.0, 1.0, ngrid)
    v = np.linspace(-1.0, 1.0, ngrid)
    U, V = np.meshgrid(u, v)
    R2 = U * U + V * V
    inside = R2 <= 1.0
    W = np.sqrt(np.clip(1.0 - R2, 0.0, None))

    # Pick front/back branch by nearest distance to SC along panel depth axis.
    d_front = np.abs(sc_hat[k] - W)
    d_back = np.abs(sc_hat[k] + W)
    depth = np.where(d_front <= d_back, W, -W)

    n = np.zeros(U.shape + (3,), dtype=float)
    n[..., i] = U
    n[..., j] = V
    n[..., k] = depth

    dot_s = n[..., 0] * sun_hat[0] + n[..., 1] * sun_hat[1] + n[..., 2] * sun_hat[2]
    dot_sc = n[..., 0] * sc_hat[0] + n[..., 1] * sc_hat[1] + n[..., 2] * sc_hat[2]

    d = n - sc_u[None, None, :]
    d_norm = np.linalg.norm(d, axis=2)
    d_norm = np.where(d_norm <= 1e-12, 1.0, d_norm)
    d_hat = d / d_norm[..., None]
    dot_t = d_hat[..., 0] * target_hat[0] + d_hat[..., 1] * target_hat[1] + d_hat[..., 2] * target_hat[2]

    mask = inside & (dot_s >= 0.0) & (dot_sc >= thr) & (dot_t >= 0.0)
    return U, V, mask


def _plot_simple_geometry(target_ra_deg: float, target_dec_deg: float, alt_km: float, mean_anomaly_offset_deg: float, earth_avoidance_default_deg: float, earth_avoidance_day_deg: float | None, earth_avoidance_night_deg: float | None, show_earth_frame: bool = True, show_xz_helpers: bool = True, time_utc: str | Time | None = None):
    sun_hat, sun_time, sun_lon_deg, sun_lat_deg = _sun_hat_now_ecliptic(time_utc=time_utc)
    target_hat = _target_from_radec(target_ra_deg, target_dec_deg)

    sc_vec_km = _sc_state_from_elements(alt_km=alt_km, mean_anomaly_offset_deg=mean_anomaly_offset_deg)
    sc_hat = _unit(sc_vec_km)
    sc_u = sc_vec_km / R_EARTH_KM
    thr = float(R_EARTH_KM / np.linalg.norm(sc_vec_km))

    n_hat = _compute_pr7_n_hat(sc_vec_km, target_hat)
    dot_ns = float(np.dot(n_hat, sun_hat))

    # Sample-based fractions using current SC state.
    rng = np.random.default_rng(0)
    n_samp = rng.normal(size=(30000, 3))
    n_samp /= np.linalg.norm(n_samp, axis=1, keepdims=True)
    visible = (n_samp @ sc_hat) >= thr
    dayside = (n_samp @ sun_hat) >= 0.0
    d = n_samp - sc_u[None, :]
    d /= np.linalg.norm(d, axis=1, keepdims=True)
    toward_target = (d @ target_hat) >= 0.0
    frac_visible = float(visible.mean())
    frac_dayside_visible = float((visible & dayside).mean())
    frac_dayside_visible_toward = float((visible & dayside & toward_target).mean())

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    planes = [(0, 1, 'X', 'Y', 'XY'), (0, 2, 'X', 'Z', 'XZ'), (1, 2, 'Y', 'Z', 'YZ')]

    viewer_by_title = {
        'XY': np.array([0.0, 0.0, 1.0], dtype=float),  # view from +Z
        'XZ': np.array([0.0, 1.0, 0.0], dtype=float),  # view from +Y
        'YZ': np.array([1.0, 0.0, 0.0], dtype=float),  # view from +X
    }

    # Sample full orbit once for panel overlays.
    orbit_dm = np.linspace(-180.0, 180.0, 2401)
    orbit_u = np.array([
        _sc_state_from_elements(alt_km=alt_km, mean_anomaly_offset_deg=float(dm)) / R_EARTH_KM
        for dm in orbit_dm
    ])

    def _plot_masked_line(ax, x, y, mask, linestyle, label=None):
        xx = np.where(mask, x, np.nan)
        yy = np.where(mask, y, np.nan)
        ax.plot(xx, yy, color='black', linewidth=0.9, linestyle=linestyle, alpha=0.9, label=label, zorder=5)

    for idx, (i, j, xlab, ylab, title) in enumerate(planes):
        ax = axes[idx]
        U, V, mask = _panel_mask(i, j, sc_hat, sc_u, sun_hat, target_hat, thr)

        shade = np.where(mask, 1.0, np.nan)
        ax.contourf(U, V, shade, levels=[0.5, 1.5], colors=['#9fd8ff'], alpha=0.55, zorder=1)

        earth = plt.Circle((0.0, 0.0), 1.0, fill=False, color='gray', linewidth=1.5, zorder=3)
        ax.add_patch(earth)

        if show_earth_frame:
            def draw_axis_arrow(vec3, color, label):
                d2 = np.array([vec3[i], vec3[j]], dtype=float)
                dn = float(np.linalg.norm(d2))
                if dn < 1e-12:
                    return
                if title == 'XY':
                    end = 0.95 * d2
                    text_pos = 1.02 * d2
                else:
                    d2 = d2 / dn
                    end = 0.95 * d2
                    text_pos = 1.02 * d2
                ax.arrow(
                    0.0,
                    0.0,
                    end[0],
                    end[1],
                    color=color,
                    linewidth=1.4,
                    linestyle=':',
                    length_includes_head=True,
                    head_width=0.035,
                    head_length=0.05,
                    label=label if idx == 1 else None,
                    zorder=7,
                )
                ax.text(text_pos[0], text_pos[1], label[0], color=color, fontsize=9, zorder=7)

            draw_axis_arrow(EARTH_SPIN_AXIS, '#8a2be2', label = 'Earth')

        # Earth equator projected into each panel.
        # The equator is the great circle n·EARTH_SPIN_AXIS = 0.
        if show_earth_frame:
            equator_pts = _great_circle_points_from_normal(EARTH_SPIN_AXIS)
            ax.plot(
                equator_pts[:, i],
                equator_pts[:, j],
                color='#8a2be2',
                linestyle='--',
                linewidth=0.8,
                alpha=0.8,
                # label='equator' if idx == 0 else None,
                zorder=3,
            )

        # Orbit overlay:
        # - dashed only where track is behind Earth *and* inside projected disk
        # - solid everywhere else (front side + outside disk silhouette)
        ox = orbit_u[:, i]
        oy = orbit_u[:, j]
        view_hat = viewer_by_title[title]
        orbit_depth = orbit_u @ view_hat
        in_disk = (ox * ox + oy * oy) <= 1.0
        hidden = (orbit_depth < 0.0) & in_disk
        shown = ~hidden
        _plot_masked_line(ax, ox, oy, shown, '-')
        _plot_masked_line(ax, ox, oy, hidden, '--')

        # Sun terminator projected as front/back branches of the great circle
        # perpendicular to sun_hat in this panel.
        term_pts = _great_circle_points_from_normal(sun_hat)
        term_depth = term_pts @ view_hat
        term_visible = term_depth >= -1e-9
        term_hidden = ~term_visible
        term_x_vis = np.where(term_visible, term_pts[:, i], np.nan)
        term_y_vis = np.where(term_visible, term_pts[:, j], np.nan)
        term_x_hid = np.where(term_hidden, term_pts[:, i], np.nan)
        term_y_hid = np.where(term_hidden, term_pts[:, j], np.nan)
        ax.plot(
            term_x_vis,
            term_y_vis,
            color='#2e5fbf',
            linewidth=2.0,
            alpha=0.9,
            label='terminator' if idx == 0 else None,
            zorder=4,
        )
        ax.plot(
            term_x_hid,
            term_y_hid,
            color='#2e5fbf',
            linewidth=1.6,
            linestyle='--',
            alpha=0.9,
            zorder=4,
        )

        S = np.array([sc_u[i], sc_u[j]], dtype=float)
        ax.scatter([S[0]], [S[1]], color='black', s=40, marker='s', label='Pandora' if idx == 0 else None, zorder=6)
        ax.text(S[0] + 0.03, S[1] + 0.03, 'Pandora', color='black', fontsize=9)

        def draw_from_sc(vec3, color, label):
            d2 = np.array([vec3[i], vec3[j]], dtype=float)
            dn = float(np.linalg.norm(d2))
            if dn < 1e-12:
                return
            # Preserve the true projected length in this panel instead of
            # renormalizing to unit length. This makes out-of-plane vectors
            # appear shorter, which is the correct 2D projection behaviour.
            end = S + 0.50 * d2
            ax.arrow(
                S[0], S[1], end[0] - S[0], end[1] - S[1],
                color=color, linewidth=2.0,
                length_includes_head=True, head_width=0.03, head_length=0.05,
                label=label if idx == 0 else None, zorder=7,
            )

        def draw_from_origin(vec3, color, label):
            d2 = np.array([vec3[i], vec3[j]], dtype=float)
            dn = float(np.linalg.norm(d2))
            if dn < 1e-12:
                return
            d2 = d2 / dn
            ax.arrow(
                0.0, 0.0, 0.55 * d2[0], 0.55 * d2[1],
                color=color, linewidth=2.0,
                length_includes_head=True, head_width=0.03, head_length=0.05,
                label=label if idx == 0 else None, zorder=7,
            )

        draw_from_sc(sun_hat, 'gold', 'to Sun')
        draw_from_sc(target_hat, 'red', 'to target')
        # draw_from_origin(n_hat, '#00bcd4', 'n (nearest-limb normal)')
        # draw_from_origin(sun_hat, '#ff8c00', 's (Sun direction)')

        if title == 'XZ' and show_xz_helpers:
            limb_xy = np.array([n_hat[i], n_hat[j]], dtype=float)
            ax.scatter([limb_xy[0]], [limb_xy[1]], color='#006400', s=30, zorder=8)
            ax.text(limb_xy[0] + 0.03, limb_xy[1] + 0.03, 'L', color='#006400', fontsize=12, zorder=9)
            ax.plot(
                [S[0], limb_xy[0]],
                [S[1], limb_xy[1]],
                color='#006400',
                linewidth=1.2,
                linestyle=':',
                alpha=0.9,
                zorder=6,
            )
            ax.plot(
                [S[0], 0.0],
                [S[1], 0.0],
                color='#666666',
                linewidth=1.2,
                linestyle=':',
                alpha=0.9,
                zorder=6,
            )
            ax.text(0.03, 0.03, 'Earth center', color='#666666', fontsize=11, zorder=9)
            target_tip = S + 0.50 * np.array([target_hat[i], target_hat[j]], dtype=float)
            ax.text(
                target_tip[0] + 0.03,
                target_tip[1] + 0.03,
                'Target',
                color='red',
                fontsize=11,
                zorder=9,
            )

        ax.axhline(0.0, color='lightgray', linewidth=0.8)
        ax.axvline(0.0, color='lightgray', linewidth=0.8)
        ax.set_aspect('equal', adjustable='box')
        if title == 'XZ':
            ax.set_xlim(1.4, -1.4)
        else:
            ax.set_xlim(-1.4, 1.4)
        ax.set_ylim(-1.4, 1.4)
        ax.set_xlabel(xlab)
        ax.set_ylabel(ylab)
        ax.set_title(title)
        ax.grid(alpha=0.25)

    axes[0].plot([], [], color='#9fd8ff', linewidth=8, alpha=0.55, label='dayside seen from Pandora toward target')
    axes[0].plot([], [], color='black', linewidth=0.9, linestyle='-')
    axes[0].plot([], [], color='black', linewidth=0.9, linestyle='--')
    axes[0].plot([], [], color='#2e5fbf', linewidth=1.6, linestyle='--')#, label='terminator (backside)')
    if show_earth_frame:
        axes[0].plot([], [], color='#8a2be2', linewidth=0.8, linestyle='--', label='Earth equator/spin axis')
    if show_xz_helpers:
        axes[0].plot([], [], marker='o', color='#006400', linestyle='None', label='nearest limb point L')
    # if show_earth_frame:
    #     # axes[0].plot([], [], color='#2e8b57', linewidth=1.0, linestyle=':', label='ecliptic north')
    #     axes[1].legend(loc='lower left', fontsize=8)
    axes[0].legend(loc='lower left', fontsize=8)

    fig.suptitle(
        # 'Geometry in XY/XZ/YZ using spacecraft orbital elements\n'
        f'visible area ≈ {100.0 * frac_visible:.1f}%   |   dayside∩visible ≈ {100.0 * frac_dayside_visible:.1f}%   |   '
        f'dayside∩visible∩toward-target ≈ {100.0 * frac_dayside_visible_toward:.1f}%\n'
        f'target(RA, Dec)=({target_ra_deg:.0f}°, {target_dec_deg:+.0f}°),  mean anomaly offset={mean_anomaly_offset_deg:+.0f}°,  obliquity={EARTH_OBLIQUITY_DEG:.2f}° (+Z=ecliptic north)\n'
        f'Sun now (geocentric ecliptic): lon={sun_lon_deg:.2f}°, lat={sun_lat_deg:+.2f}°, UTC={sun_time.isot[:19]}'
    )
    plt.tight_layout()
    return fig


In [11]:
# --- Interactive run cell (requires ipywidgets) ---
import ipywidgets as widgets
from IPython.display import display

# Clean up prior widget instances if this cell is re-run.
for _name in [
    'ma_offset_slider',
    'target_ra_slider',
    'target_dec_slider',
    'alt_slider',
    'show_earth_frame_checkbox',
    'show_xz_helpers_checkbox',
    'use_now_checkbox',
    'utc_time_text',
    'earth_default_text',
    'use_daynight_checkbox',
    'earth_day_text',
    'earth_night_text',
    'controls',
    'interactive_out',
]:
    if _name in globals():
        try:
            globals()[_name].close()
        except Exception:
            pass

ma_offset_slider = widgets.FloatSlider(value=145.0, min=-180.0, max=180.0, step=1.0, description='dM [deg]')
target_ra_slider = widgets.FloatSlider(value=145.0, min=0.0, max=360.0, step=1.0, description='Target RA')
target_dec_slider = widgets.FloatSlider(value=10.0, min=-90.0, max=90.0, step=1.0, description='Target Dec')
alt_slider = widgets.FloatSlider(value=600.0, min=400.0, max=2000.0, step=10.0, description='Alt [km]')
show_earth_frame_checkbox = widgets.Checkbox(value=False, description='Show Earth equator/spin axis')
show_xz_helpers_checkbox = widgets.Checkbox(value=False, description='Show Earth-center/limb')
use_now_checkbox = widgets.Checkbox(value=True, description='Use time = now')
utc_time_text = widgets.Text(value=Time.now().utc.isot[:19], description='UTC time')
earth_default_text = widgets.FloatText(value=86.0, description='Earth default')
use_daynight_checkbox = widgets.Checkbox(value=False, description='Use day/night Earth')
earth_day_text = widgets.FloatText(value=110.0, description='Earth day')
earth_night_text = widgets.FloatText(value=86.0, description='Earth night')

def _render(
    target_ra_deg,
    target_dec_deg,
    alt_km,
    mean_anomaly_offset_deg,
    show_earth_frame,
    show_xz_helpers,
    use_now,
    utc_time,
    earth_avoidance_default_deg,
    use_daynight,
    earth_avoidance_day_deg,
    earth_avoidance_night_deg,
):
    utc_time_text.disabled = bool(use_now)
    earth_day_text.disabled = not use_daynight
    earth_night_text.disabled = not use_daynight
    time_arg = None if use_now else str(utc_time).strip()
    with plt.ioff():
        fig = _plot_simple_geometry(
            target_ra_deg=target_ra_deg,
            target_dec_deg=target_dec_deg,
            alt_km=alt_km,
            mean_anomaly_offset_deg=mean_anomaly_offset_deg,
            earth_avoidance_default_deg=earth_avoidance_default_deg,
            earth_avoidance_day_deg=(earth_avoidance_day_deg if use_daynight else None),
            earth_avoidance_night_deg=(earth_avoidance_night_deg if use_daynight else None),
            show_earth_frame=show_earth_frame,
            show_xz_helpers=show_xz_helpers,
            time_utc=time_arg,
        )
    display(fig)
    plt.close(fig)

controls = widgets.VBox([
    widgets.HBox([target_ra_slider, target_dec_slider]),
    widgets.HBox([alt_slider, ma_offset_slider]),
    widgets.HBox([show_earth_frame_checkbox, show_xz_helpers_checkbox]),
    widgets.HBox([use_now_checkbox, utc_time_text]),
    widgets.HBox([earth_default_text, use_daynight_checkbox]),
    widgets.HBox([earth_day_text, earth_night_text]),
])

interactive_out = widgets.interactive_output(
    _render,
    {
        'target_ra_deg': target_ra_slider,
        'target_dec_deg': target_dec_slider,
        'alt_km': alt_slider,
        'mean_anomaly_offset_deg': ma_offset_slider,
        'show_earth_frame': show_earth_frame_checkbox,
        'show_xz_helpers': show_xz_helpers_checkbox,
        'use_now': use_now_checkbox,
        'utc_time': utc_time_text,
        'earth_avoidance_default_deg': earth_default_text,
        'use_daynight': use_daynight_checkbox,
        'earth_avoidance_day_deg': earth_day_text,
        'earth_avoidance_night_deg': earth_night_text,
    },
)

display(controls, interactive_out)


Output()